In [14]:
import sys, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from collections import defaultdict

from pathlib     import Path

from torch.utils.data       import (DataLoader, 
                                    TensorDataset, 
                                    WeightedRandomSampler)

from sklearn.metrics        import (balanced_accuracy_score, 
                                    accuracy_score, 
                                    confusion_matrix)

from sklearn.decomposition import PCA

sys.path.append('..')
import module

from src.model.modules    import (MicroExpressionDataset, 
                                  Normalizer, 
                                  SpatialTemporalCNN, 
                                  FeatureExtractor)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [15]:
# ── Load dataset & identifikasi subjek fold 0 ─────────────────────────────
ANNOTATIONS = os.path.join(Path.home(), "datasets", "anxiety_raw", "annotations-v2.csv")

dataset = MicroExpressionDataset(
    annotations_file=ANNOTATIONS, stage='all',
    mode='eval', valid_only=True)

folds   = MicroExpressionDataset.build_group_kfold(dataset.annotations, n_splits=5)
fold0   = folds[0]

train_idx = fold0['train_idx']
test_idx  = fold0['test_idx']

ann_test  = dataset.annotations.iloc[test_idx].reset_index(drop=True)
ann_train = dataset.annotations.iloc[train_idx].reset_index(drop=True)

test_subjects_full = ann_test.drop_duplicates('subject_id')[[
    'subject_id','anxiety_level','anxiety_score',
    'anxiety_category','gender','age'
]].reset_index(drop=True)

print('=' * 60)
print('Subjek TEST di Fold 0')
print('=' * 60)
print(test_subjects_full.to_string(index=False))
print(f'\nJumlah clip test: {len(ann_test)}')
print(f'Distribusi stage: {ann_test["stage"].value_counts().to_dict()}')
print(f'Distribusi label: {ann_test["anxiety_level"].value_counts().to_dict()}')

Subjek TEST di Fold 0
                subject_id anxiety_level  anxiety_score  anxiety_category gender  age
  ahmad_rifqi_hendriansyah          high              0               NaN    man    0
     fabian_ananda_merdana          high              0               NaN    man    0
             satria_wiguna          high              0               NaN    man    0
      oltha_rosyeda_al_haq          high              0               NaN    man    0
    akhmad_aakhif_athallah           low              0               NaN    man    0
    ashrul_rifki_ardiyhasa           low              0               NaN    man    0
     haikal_muhammad_rafli           low              0               NaN    man    0
      mohammad_adri_favian           low              0               NaN    man    0
kamila_habiba_putri_ananta           low              0               NaN    man    0
             muhammad_zaki           low              0               NaN    man    0
         vira_alfita_yunia      

In [16]:
# ── Perbandingan karakteristik: test fold 0 vs training ──────────────────
print('=' * 60)
print('Perbandingan Karakteristik Test vs Training')
print('=' * 60)

train_subj = ann_train.drop_duplicates('subject_id')
test_subj  = ann_test.drop_duplicates('subject_id')

for col in ['anxiety_score','age']:
    t_mean = train_subj[col].mean() if col in train_subj.columns else None
    te_mean = test_subj[col].mean()  if col in test_subj.columns else None
    if t_mean is not None:
        print(f'{col:20s}: train={t_mean:.2f}  test={te_mean:.2f}')

print(f'{"gender":20s}: train={dict(train_subj["gender"].value_counts())}  '
      f'test={dict(test_subj["gender"].value_counts())}')

print(f'{"anxiety_category":20s}: train={dict(train_subj["anxiety_category"].value_counts())}  '
      f'test={dict(test_subj["anxiety_category"].value_counts())}')

print()
print('Anxiety score test subjects:')
print(test_subj[['subject_id','anxiety_level','anxiety_score',
                  'anxiety_category']].to_string(index=False))

Perbandingan Karakteristik Test vs Training
anxiety_score       : train=0.00  test=0.00
age                 : train=0.00  test=0.00
gender              : train={'man': np.int64(44)}  test={'man': np.int64(12)}
anxiety_category    : train={}  test={}

Anxiety score test subjects:
                subject_id anxiety_level  anxiety_score  anxiety_category
  ahmad_rifqi_hendriansyah          high              0               NaN
     fabian_ananda_merdana          high              0               NaN
             satria_wiguna          high              0               NaN
      oltha_rosyeda_al_haq          high              0               NaN
    akhmad_aakhif_athallah           low              0               NaN
    ashrul_rifki_ardiyhasa           low              0               NaN
     haikal_muhammad_rafli           low              0               NaN
      mohammad_adri_favian           low              0               NaN
kamila_habiba_putri_ananta           low              

In [17]:
# ── Analisis fitur: posisi test fold 0 dalam feature space ───────────────
print('=' * 60)
print('Posisi Fitur: Test Fold 0 vs Training')
print('=' * 60)

dataset.set_mode('eval'); dataset.clear_cache()
X_all, y_all, subs_all = dataset.get_all_features()

X_tr = X_all[train_idx]
y_tr = y_all[train_idx]
X_te = X_all[test_idx]
y_te = y_all[test_idx]

norm   = Normalizer()
X_tr_n = norm.fit_transform(X_tr)
X_te_n = norm.transform(X_te)

# PCA 2D untuk visualisasi
pca    = PCA(n_components=2)
X_tr_p = pca.fit_transform(X_tr_n)
X_te_p = pca.transform(X_te_n)

var = pca.explained_variance_ratio_
print(f'PCA variance: PC1={var[0]:.3f}, PC2={var[1]:.3f}')

# Jarak centroid test dari centroid training per kelas
for cls, name in [(0,'LOW'), (1,'HIGH')]:
    tr_c    = X_tr_p[y_tr == cls].mean(axis=0)
    te_c    = X_te_p[y_te == cls].mean(axis=0) if (y_te == cls).any() else None
    if te_c is not None:
        dist = np.linalg.norm(te_c - tr_c)
        print(f'{name}: centroid dist test-dari-train = {dist:.4f}')

# Apakah test outlier dari distribusi training?
from scipy.spatial.distance import cdist
for cls, name in [(0,'LOW'), (1,'HIGH')]:
    tr_cls = X_tr_n[y_tr == cls]
    te_cls = X_te_n[y_te == cls] if (y_te == cls).any() else None
    if te_cls is not None and len(te_cls) > 0:
        dists   = cdist(te_cls, tr_cls, metric='euclidean').min(axis=1)
        tr_self = cdist(tr_cls, tr_cls, metric='euclidean')
        np.fill_diagonal(tr_self, np.inf)
        tr_nn   = tr_self.min(axis=1).mean()
        print(f'{name}: avg min-dist test-ke-training = {dists.mean():.4f} '
              f'(train inter-sample avg = {tr_nn:.4f})')

Posisi Fitur: Test Fold 0 vs Training
PCA variance: PC1=0.048, PC2=0.042
LOW: centroid dist test-dari-train = 0.1125
HIGH: centroid dist test-dari-train = 0.9300
LOW: avg min-dist test-ke-training = 8.6018 (train inter-sample avg = 8.3204)
HIGH: avg min-dist test-ke-training = 7.9497 (train inter-sample avg = 8.8646)


In [18]:
# ── Re-train fold 0 dengan 3 strategi dan analisis epoch-by-epoch ────────
print('=' * 60)
print('Re-train Fold 0 — Tracking per Epoch (3 Strategi)')
print('=' * 60)

EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 32
PATIENCE     = 20
criterion    = nn.CrossEntropyLoss(label_smoothing=0.05)

counts  = np.bincount(y_tr, minlength=2)
weights = torch.tensor([1./counts[l] for l in y_tr], dtype=torch.float)
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
tr_ds   = TensorDataset(torch.from_numpy(X_tr_n),
                         torch.from_numpy(y_tr).long(),
                         torch.arange(len(X_tr_n)))
te_ds   = TensorDataset(torch.from_numpy(X_te_n),
                         torch.from_numpy(y_te).long(),
                         torch.arange(len(X_te_n)))
tr_ld   = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)
te_ld   = DataLoader(te_ds, batch_size=len(X_te_n), shuffle=False)

subs_test = [subs_all[i] for i in test_idx]

history = {tag: {'loss':[], 'val_uar':[], 'ep_stopped': EPOCHS-1}
           for tag in ['A','B','C']}

models  = {tag: SpatialTemporalCNN().to(device) for tag in ['A','B','C']}
opts    = {tag: torch.optim.Adam(models[tag].parameters(),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
           for tag in ['A','B','C']}
schs    = {tag: torch.optim.lr_scheduler.CosineAnnealingLR(
               opts[tag], T_max=EPOCHS, eta_min=1e-6)
           for tag in ['A','B','C']}
scls    = {tag: torch.amp.GradScaler('cuda') for tag in ['A','B','C']}

state   = {tag: {'best_loss': float('inf'), 'best_uar': -1.,
                  'ckpt': None, 'no_imp': 0, 'stopped': False}
           for tag in ['A','B','C']}

for epoch in range(EPOCHS):
    if all(state[t]['stopped'] for t in ['A','B','C']): break

    for tag in ['A','B','C']:
        if state[tag]['stopped']: continue
        m = models[tag]; m.train()
        ep_loss = 0.
        for feat, label, _ in tr_ld:
            feat, label = feat.to(device), label.to(device)
            opts[tag].zero_grad()
            with torch.amp.autocast('cuda'):
                loss = criterion(m(feat), label)
            scls[tag].scale(loss).backward()
            scls[tag].unscale_(opts[tag])
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scls[tag].step(opts[tag]); scls[tag].update()
            ep_loss += loss.item()
        schs[tag].step()
        ep_loss /= max(len(tr_ld), 1)

        m.eval()
        with torch.no_grad():
            feat_te, lbl_te, _ = next(iter(te_ld))
            p  = torch.softmax(m(feat_te.to(device)),dim=1).cpu().numpy()
            pr = np.argmax(p, axis=1)
        val_uar = balanced_accuracy_score(lbl_te.numpy(), pr)

        history[tag]['loss'].append(ep_loss)
        history[tag]['val_uar'].append(val_uar)

        s = state[tag]
        if tag in ('B','C') and val_uar > s['best_uar']:
            s['best_uar'] = val_uar
            s['ckpt']     = {k:v.cpu().clone() for k,v in m.state_dict().items()}
            if tag == 'B': s['no_imp'] = 0
        if tag in ('A','C'):
            if ep_loss < s['best_loss'] - 1e-4:
                s['best_loss'] = ep_loss; s['no_imp'] = 0
                if tag == 'A':
                    s['ckpt'] = {k:v.cpu().clone() for k,v in m.state_dict().items()}
            else:
                s['no_imp'] += 1
        if tag == 'B' and val_uar <= s['best_uar'] and epoch > 0:
            s['no_imp'] += 1
        if s['no_imp'] >= PATIENCE:
            s['stopped'] = True
            history[tag]['ep_stopped'] = epoch

print('Epoch history siap.')
for tag in ['A','B','C']:
    ep_s = history[tag]['ep_stopped']
    best_uar = max(history[tag]['val_uar'])
    print(f'  [{tag}] stopped=ep{ep_s} | best_val_uar={best_uar:.4f}')

Re-train Fold 0 — Tracking per Epoch (3 Strategi)
Epoch history siap.
  [A] stopped=ep90 | best_val_uar=0.5606
  [B] stopped=ep20 | best_val_uar=0.5221
  [C] stopped=ep78 | best_val_uar=0.6366


In [19]:
# ── Evaluasi final fold 0 per strategi ───────────────────────────────────
print('=' * 60)
print('Evaluasi Final Fold 0 — Detail per Strategi')
print('=' * 60)

feat_te, lbl_te, idx_te = next(iter(te_ld))

for tag in ['A','B','C']:
    s = state[tag]
    if s['ckpt']:
        models[tag].load_state_dict(s['ckpt'])
    models[tag].eval()
    with torch.no_grad():
        p   = torch.softmax(models[tag](feat_te.to(device)),dim=1).cpu().numpy()
        pr  = np.argmax(p, axis=1)
    lnp = lbl_te.numpy()
    uar = balanced_accuracy_score(lnp, pr)
    cm  = confusion_matrix(lnp, pr, labels=[0,1])
    print(f'\nStrategi {tag} | stopped=ep{history[tag]["ep_stopped"]} | UAR={uar:.4f}')
    print(f'  Confusion: {cm.tolist()}')
    print(f'  Per-clip:')
    for i, si in enumerate(idx_te.tolist()):
        ann_row = ann_test.iloc[si]
        correct = '✓' if pr[i] == lnp[i] else '✗'
        print(f'    {correct} {ann_row["subject_id"]:40s} '
              f'clip={ann_row["clip"]} stage={ann_row["stage"]:6s} '
              f'true={lnp[i]} pred={pr[i]} prob_high={p[i][1]:.3f}')

# Kurva UAR per epoch untuk fold 0
print('\nVal UAR per epoch (setiap 5):')
header = f'{"ep":>4}' + ''.join([f'  {t}_uar' for t in ['A','B','C']])
print(header)
max_ep = max(len(history[t]['val_uar']) for t in ['A','B','C'])
for ep in range(0, max_ep, 5):
    row = f'{ep:>4}'
    for t in ['A','B','C']:
        v = history[t]['val_uar'][ep] if ep < len(history[t]['val_uar']) else '-'
        row += f'  {v:.4f}' if isinstance(v, float) else f'  {v:>6}'
    print(row)

Evaluasi Final Fold 0 — Detail per Strategi

Strategi A | stopped=ep90 | UAR=0.4913
  Confusion: [[51, 43], [14, 11]]
  Per-clip:
    ✓ ahmad_rifqi_hendriansyah                 clip=q3 stage=before true=1 pred=1 prob_high=0.638
    ✗ ahmad_rifqi_hendriansyah                 clip=q4 stage=before true=1 pred=0 prob_high=0.494
    ✓ ahmad_rifqi_hendriansyah                 clip=q5 stage=before true=1 pred=1 prob_high=0.602
    ✓ ahmad_rifqi_hendriansyah                 clip=q1 stage=before true=1 pred=1 prob_high=0.542
    ✗ ahmad_rifqi_hendriansyah                 clip=q2 stage=before true=1 pred=0 prob_high=0.394
    ✗ fabian_ananda_merdana                    clip=q3 stage=before true=1 pred=0 prob_high=0.472
    ✓ fabian_ananda_merdana                    clip=q4 stage=before true=1 pred=1 prob_high=0.776
    ✗ fabian_ananda_merdana                    clip=q5 stage=before true=1 pred=0 prob_high=0.412
    ✗ fabian_ananda_merdana                    clip=q1 stage=before true=1 pred=0 prob

In [20]:
# ── Ringkasan: hipotesis mengapa fold 0 berperilaku berbeda ──────────────
print('=' * 60)
print('Ringkasan Analisis Fold 0')
print('=' * 60)

print()
print('Subjek test fold 0:')
print(test_subjects_full[['subject_id','anxiety_level','anxiety_score', 'anxiety_category']].to_string(index=False))

print()
print('Hasil per strategi:')
print('  A: UAR rendah  — checkpoint dari train_loss tidak merepresentasikan')
print('     generalisasi. Model berhenti terlalu terlambat di posisi yang salah.')
print('  B: UAR sedang  — berhenti terlalu cepat (ep ~20), model belum matang')
print('     untuk pola subjek fold 0.')
print('  C: UAR tinggi  — model diizinkan terus belajar (train_loss) sambil')
print('     menyimpan titik terbaik yang pernah dicapai di validation.')
print('     Subjek fold 0 kemungkinan butuh lebih banyak epoch untuk konverge.')

print()
print('Hipotesis untuk diverifikasi dari data di atas:')
print('  1. Apakah anxiety_score test subjects sangat berbeda dari train mean?')
print('  2. Apakah anxiety_category (naik/turun/tetap) berbeda dari mayoritas train?')
print('  3. Apakah centroid distance test jauh dari distribusi train?')

Ringkasan Analisis Fold 0

Subjek test fold 0:
                subject_id anxiety_level  anxiety_score  anxiety_category
  ahmad_rifqi_hendriansyah          high              0               NaN
     fabian_ananda_merdana          high              0               NaN
             satria_wiguna          high              0               NaN
      oltha_rosyeda_al_haq          high              0               NaN
    akhmad_aakhif_athallah           low              0               NaN
    ashrul_rifki_ardiyhasa           low              0               NaN
     haikal_muhammad_rafli           low              0               NaN
      mohammad_adri_favian           low              0               NaN
kamila_habiba_putri_ananta           low              0               NaN
             muhammad_zaki           low              0               NaN
         vira_alfita_yunia           low              0               NaN
       fandy_wahyu_hanzura           low              0          